Configure the local data snapshots as described in the README before running this notebook.

# BOVA11 volatility and risk research lab

Forecasts target the next exchange session. Configure the private snapshot as described in the README.

In [ ]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable,"-m","pip","-q","install","arch","hmmlearn","xgboost","exchange-calendars"])

In [ ]:
from local_snapshots import load_local_snapshot

SNAPSHOTS = load_local_snapshot('BOVA11_Volatility_Research_Lab.ipynb', 'SNAPSHOTS')


## 1. Data and forecast horizon

The archived data span November 2020 through 5 November 2026. BOVA11 uses adjusted closing prices. Set `ATIVO` to `IBOV` to analyze the index.

In [ ]:
import base64, io, zlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy import stats
from scipy.special import logsumexp
from statsmodels.stats.diagnostic import acorr_ljungbox
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

ASSET = "BOVA11"          # BOVA11 or IBOV
START_TEST = "2025-01-01"
CSV_RV = globals().get("CSV_RV")             # Optional Date,RV file with validated daily realized variance from intraday bars.
OUTPUT = Path("results_risk")/ASSET
OUTPUT.mkdir(parents=True,exist_ok=True)
EPS = 1e-10
SEED = 42
STEP_REFIT = 63

def read_snapshot(name):
    raw = zlib.decompress(base64.b64decode(SNAPSHOTS[name]))
    df = pd.read_csv(io.BytesIO(raw),index_col="Date",parse_dates=True)
    df.index = df.index.astype("datetime64[ns]")
    assert df.index.is_unique and df.index.is_monotonic_increasing
    return df

assert ASSET in {"BOVA11","IBOV"}
ibov = read_snapshot("ibov")
try:
    from market_calendar import calendar_and_sessions
except ModuleNotFoundError as error:
    if error.name != "market_calendar":
        raise
    import exchange_calendars as xcals

    # Fallback generated from market_calendar.py.
    def calendar_and_sessions(name, start, end):
        """Return the exchange calendar and session labels between two inclusive dates."""
        start, end = pd.Timestamp(start), pd.Timestamp(end)
        if any(pd.isna(day) or day.tz is not None or day != day.normalize()
               for day in (start, end)) or start > end:
            raise ValueError("Enter local dates without times in ascending order.")
        margin = pd.Timedelta(days=15)
        calendar = xcals.get_calendar(name, start=start - margin, end=end + margin)
        sessions = calendar.sessions_in_range(start, end)
        if sessions.tz is not None:
            sessions = sessions.tz_localize(None)
        return calendar, sessions

cal, sessions = calendar_and_sessions("BVMF", ibov.index.min(), ibov.index.max())
raw = read_snapshot({"BOVA11":"bova11","IBOV":"ibov"}[ASSET])
assert raw.index.isin(sessions).all()
column = "Adj Close" if ASSET=="BOVA11" else "Close"
price = raw[column].reindex(sessions)
assert np.isfinite(price.dropna()).all() and price.dropna().gt(0).all()
r = np.log(price).diff()
variance_obs = r.pow(2)
origin = pd.Series(sessions,index=sessions)
target_date = origin.shift(-1)
first_test = pd.Timestamp(START_TEST)
test = sessions[(target_date>=first_test).to_numpy()]
origins_forecast = test.append(pd.DatetimeIndex([sessions[-1]]))
assert len(test)>=40
def load_rv_validated(path, asset, sessions, first_test, test):
    import json

    manifest = Path(path).with_name("manifest_rv.json")
    if manifest.exists():
        provenance = json.loads(manifest.read_text(encoding="utf-8"))
        if provenance["asset"] != asset:
            raise ValueError("The realized variance series is for a different asset.")
    table = pd.read_csv(path, index_col="Date", parse_dates=True)
    if table.empty:
        raise ValueError("The RV CSV contains no approved sessions.")
    if (not isinstance(table.index,pd.DatetimeIndex) or not table.index.is_unique
        or not table.index.isin(sessions).all()):
        raise ValueError("RV requires unique dates within the snapshot calendar.")
    rv = pd.to_numeric(table.RV, errors="raise").reindex(sessions)
    if not np.isfinite(rv.dropna()).all() or rv.dropna().lt(0).any():
        raise ValueError("RV must be finite and nonnegative.")
    windows = rv.rolling(22).count().eq(22)
    if windows.loc[windows.index < first_test].sum() < 100:
        raise ValueError("RV needs at least 100 valid 22-session windows before the test period.")
    if not (windows & rv.shift(-1).notna()).reindex(test,fill_value=False).any():
        raise ValueError("RV needs at least one test origin with 22 sessions and an observed next-session target.")
    return rv


rv = load_rv_validated(CSV_RV, ASSET, sessions, first_test, test) if CSV_RV is not None else None

print(f"{ASSET} | {sessions[0].date()} through {sessions[-1].date()} | {price.isna().sum()} sessions without prices")
print("Dates without prices:", [str(d.date()) for d in price.index[price.isna()]])
print("Computed returns:",r.notna().sum(),"| first test origins:",test[0].date())

## 2. Historical volatility, drawdown, and tails

Drawdown is the ratio of the observed price to its running peak, minus one. Annualized rolling volatility uses 252 sessions. Autocorrelations are descriptive.

In [ ]:
equity = price/price.dropna().iloc[0]
drawdown = equity/equity.cummax()-1
worst = drawdown.idxmin()
peak = equity.loc[:worst].idxmax()
recoveries = equity.loc[worst:].dropna()
recoveries = recoveries.loc[recoveries>=equity.loc[peak]]
recuperacao = recoveries.index[0] if len(recoveries) else pd.NaT
clean_returns = r.dropna()
statistics = pd.Series({"Annualized volatility":clean_returns.std()*np.sqrt(252),
                  "maximum drawdown":drawdown.min(),"Peak date":peak.date(),
                  "Trough date":worst.date(),"recovery":recuperacao,
                  "return skewness":clean_returns.skew(),"excess kurtosis":clean_returns.kurt()})
display(statistics)
fig,axs = plt.subplots(2,2,figsize=(13,7))
equity.plot(ax=axs[0,0],title="Equity per unit of initial capital")
drawdown.plot(ax=axs[0,1],title="Drawdown from the latest peak",color="firebrick")
axs[0,1].yaxis.set_major_formatter(PercentFormatter(1))
r.plot(ax=axs[1,0],title="Daily log returns",lw=.65)
axs[1,0].yaxis.set_major_formatter(PercentFormatter(1))
r.rolling(21).std().mul(np.sqrt(252)).plot(ax=axs[1,1],title="Annualized rolling volatility over 21 sessions")
axs[1,1].yaxis.set_major_formatter(PercentFormatter(1))
fig.tight_layout()
fig.savefig(OUTPUT/"panel_historical.png",dpi=140)
plt.show()
# Use a contiguous block for ACF so sessions separated by gaps are never treated as adjacent.
blocks = [s.dropna() for _,s in r.groupby(r.isna().cumsum())]
contiguous = max(blocks,key=len)
from statsmodels.graphics.tsaplots import plot_acf
fig,ax = plt.subplots(1,2,figsize=(12,3.5))
plot_acf(contiguous,lags=25,ax=ax[0],title="ACF of returns")
plot_acf(contiguous.pow(2),lags=25,ax=ax[1],title="ACF of squared returns")
plt.tight_layout()
plt.show()
display(acorr_ljungbox(contiguous.pow(2),lags=[5,10,20],return_df=True))
drawdown.rename("drawdown").to_csv(OUTPUT/"drawdown.csv",index_label="data")

## 3. Baselines, GARCH, EGARCH, GJR, and HMM

The historical baseline averages 21 squared returns. EWMA uses 0.94 decay. GARCH-family fits use zero mean and Student-t innovations.

In [ ]:
forecasts = pd.DataFrame(index=sessions)
forecasts["Historical 21"] = variance_obs.rolling(21).mean()
# Restart EWMA after a missing return and leave the gap unfilled.
forecasts["EWMA 0.94"] = variance_obs.groupby(r.isna().cumsum()).transform(
    lambda s:s.ewm(alpha=.06,adjust=False,min_periods=21).mean()).where(r.notna())

historical_pre = r.loc[r.index<first_test]
sample = max([s.dropna() for _,s in historical_pre.groupby(historical_pre.isna().cumsum())],key=len)
assert len(sample)>=252, "The initial fit requires 252 consecutive returns."
sample_pct = sample*100
config_garch = {"GARCH(1,1)":("GARCH",0),"EGARCH":("EGARCH",1),"GJR-GARCH":("GARCH",1)}
params = {}
diagnosticos = []
for name,(familia,skewness) in config_garch.items():
    fit = arch_model(sample_pct,mean="Zero",vol=familia,p=1,o=skewness,q=1,dist="t").fit(
        disp="off",show_warning=False)
    if fit.convergence_flag:
        raise RuntimeError(f"{name} did not converge on the initial fit.")
    params[name] = fit.params
    diagnosticos.append({"model":name,"n_fit":len(sample),"end_fit":sample.index.max().date(),
                         "nu":fit.params["nu"]})

# A forecast at close t uses returns observed through t.
# Use the latest contiguous block to restart the filter after gaps.
r_pct = r*100
for name,(familia,skewness) in config_garch.items():
    output = {}
    for _,block in r_pct.groupby(r.isna().cumsum()):
        block = block.dropna()
        for day in block.index.intersection(origins_forecast):
            hist = block.loc[:day]
            if len(hist)<22:
                continue
            model = arch_model(hist.to_numpy(),mean="Zero",vol=familia,p=1,o=skewness,q=1,dist="t")
            fit = model.fix(params[name])
            output[day] = fit.forecast(horizon=1,reindex=False).variance.iloc[-1,0]/1e4
    forecasts[name] = pd.Series(output,dtype=float)
display(pd.DataFrame(diagnosticos))

hmm = GaussianHMM(n_components=2,covariance_type="diag",n_iter=250,random_state=SEED).fit(
    sample_pct.to_numpy()[:,None])
if not hmm.monitor_.converged:
    warnings.warn("HMM convergence was not established. Treat the result as experimental.")
mean_state = hmm.means_.ravel()
var_state = np.maximum(hmm.covars_.reshape(2,-1)[:,0],1e-8)
prob = hmm.startprob_
second_moment,regime = [],[]
for obs in r_pct:
    if pd.notna(obs):
        logp = np.log(np.maximum(prob,1e-300))-.5*(np.log(var_state)+(obs-mean_state)**2/var_state)
        prob = np.exp(logp-logsumexp(logp))
    prob = prob @ hmm.transmat_
    second_moment.append(prob @ (var_state+mean_state**2)/1e4)
    regime.append(prob[np.argmax(var_state)])
forecasts["HMM 2 states"] = second_moment
prob_regime = pd.Series(regime,index=sessions).where(sessions>=first_test-pd.Timedelta(days=7))

## 4. Walk-forward HAR and XGBoost

HAR combines variance proxies at 1, 5, and 22 sessions. XGBoost adds recent returns and external variables dated strictly before the forecast origin. Both are refitted every 63 origins.

In [ ]:
def fatores_har(series):
    return pd.DataFrame({f"m{w}":series.rolling(w).mean() for w in [1,5,22]})

def arvores():
    return XGBRegressor(objective="reg:gamma",n_estimators=160,max_depth=2,min_child_weight=12,
                        learning_rate=.035,subsample=.85,colsample_bytree=.85,
                        random_state=SEED,n_jobs=2)

def walk_forward(series,extras,origins):
    h = fatores_har(series)
    target = series.shift(-1)
    x = h.join(extras)
    har,boost = {},{}
    for start in range(0,len(origins),STEP_REFIT):
        block = origins[start:start+STEP_REFIT]
        # The final training target must be known at the first close of the forecast block.
        train_ok = x.notna().all(axis=1)&target.notna()&(target_date<=block[0])
        idx_train = x.index[train_ok]
        idx_pred = block.intersection(x.index[x.notna().all(axis=1)])
        if len(idx_train)<100 or not len(idx_pred):
            continue
        assert target_date.loc[idx_train].max()<=idx_pred.min()
        yy = target.loc[idx_train].clip(lower=EPS)
        # HAR uses levels and a positive floor. Training and forecasts share the same scale.
        lin = LinearRegression().fit(h.loc[idx_train]*1e4,yy*1e4)
        xgb = arvores().fit(x.loc[idx_train],yy*1e4)
        har.update(zip(idx_pred,np.maximum(lin.predict(h.loc[idx_pred]*1e4)/1e4,EPS)))
        boost.update(zip(idx_pred,np.maximum(xgb.predict(x.loc[idx_pred])/1e4,EPS)))
    return pd.DataFrame({"HAR":pd.Series(har,dtype=float),"XGBoost":pd.Series(boost,dtype=float)})

extras = pd.DataFrame({"return_value":r,"return_value_negativo":r.clip(upper=0),
                       "abs_ret_5":r.abs().rolling(5).mean()},index=sessions)
for name in ["ibov","sp500","usd","vix","oil","ewz"]:
    if ASSET=="IBOV" and name=="ibov":
        continue
    external = read_snapshot(name).Close.dropna()
    assert external.gt(0).all()
    features = pd.DataFrame({f"{name}_ret1":np.log(external).diff(),
                             f"{name}_vol5":np.log(external).diff().rolling(5).std(),
                             "source":external.index})
    align = pd.merge_asof(pd.DataFrame(index=sessions),features,left_index=True,right_index=True,
                           allow_exact_matches=False,tolerance=pd.Timedelta(days=7))
    assert (align.source.dropna()<align.source.dropna().index).all()
    extras = extras.join(align.drop(columns="source"))
ml = walk_forward(variance_obs,extras,origins_forecast)
forecasts["HAR · daily proxy"] = ml.HAR
forecasts["XGBoost"] = ml.XGBoost

## 5. Model comparison

The next-session target is the squared log return, a noisy variance proxy. Models are compared on common dates. QLIKE is log predicted variance plus observed variance divided by predicted variance.

In [ ]:
def compare(forecasts,observed,origins):
    frame = forecasts.reindex(origins).join(observed.shift(-1).rename("observed"))
    disponiveis = frame.notna().sum()
    common = frame.dropna()
    if common.empty:
        raise ValueError("The models share no evaluation dates.")
    vv = common.observed.to_numpy()
    rows = []
    for name in forecasts:
        h = common[name].clip(lower=EPS).to_numpy()
        rows.append({"model":name,"QLIKE":np.mean(np.log(h)+vv/h),
                       "MAE_var":np.mean(np.abs(h-vv)),"RMSE_var":np.sqrt(np.mean((h-vv)**2)),
                       "observed_predicted":np.sum(vv)/np.sum(h),
                       "n_available":int(disponiveis[name]),"common_count":len(common)})
    return pd.DataFrame(rows).sort_values("QLIKE"),common

ranking,common = compare(forecasts,variance_obs,test)
display(ranking.style.format({"QLIKE":"{:.3f}","MAE_var":"{:.3e}","RMSE_var":"{:.3e}",
                              "observed_predicted":"{:.2f}"}))
ranking.to_csv(OUTPUT/"ranking_volatility.csv",index=False)
forecasts_export = forecasts.loc[test].join(target_date.rename("target_date")).join(
    variance_obs.shift(-1).rename("proxy_var_observed"))
forecasts_export.to_csv(OUTPUT/"next_session_forecasts.csv",index_label="origin")
dates_plot = pd.DatetimeIndex(target_date.loc[test])
common_plot = common.reindex(test)
fig,ax = plt.subplots(3,1,figsize=(13,9),sharex=True)
ax[0].plot(dates_plot,np.sqrt(common_plot.observed*252),label="Observed next-session proxy",color="0.65",lw=.65)
for name in ["Historical 21","GARCH(1,1)","HAR · daily proxy","XGBoost"]:
    ax[0].plot(dates_plot,np.sqrt(common_plot[name].clip(lower=EPS)*252),label=name,lw=1)
ax[0].set_title("Annualized volatility by target date")
ax[0].yaxis.set_major_formatter(PercentFormatter(1))
ax[0].legend(ncol=3)
ax[1].plot(dates_plot,prob_regime.reindex(test),color="#805ad5")
ax[1].set_title("HMM filtered probability of higher volatility next session")
ax[1].set_ylim(0,1)
ax[1].yaxis.set_major_formatter(PercentFormatter(1))
for name in ["EWMA 0.94","GARCH(1,1)","XGBoost"]:
    h = common[name].clip(lower=EPS)
    ax[2].plot(dates_plot,(np.log(h)+common.observed/h).rolling(42).mean().reindex(test),label=name)
ax[2].set_title("Mean QLIKE over 42 forecasts, where lower is better")
ax[2].legend()
fig.tight_layout()
fig.savefig(OUTPUT/"panel_forecasts.png",dpi=140)
plt.show()

## 6. VaR and Expected Shortfall

Loss is the negative log return. VaR is a loss quantile. Expected Shortfall averages the tail, with fractional mass at the empirical quantile boundary.

In [ ]:
# Use log-return units for losses. VaR is the quantile and CVaR is the upper-tail mean.
loss = -r
loss_d1 = loss.shift(-1)
def es_empirico(v,alpha):
    v = np.sort(v[np.isfinite(v)])[::-1]
    mass = len(v)*alpha
    integers = int(np.floor(mass))
    return (v[:integers].sum()+(mass-integers)*v[integers])/mass

risk = pd.DataFrame(index=sessions)
results_tail = []
nu = float(params["GARCH(1,1)"]["nu"])
sigma = np.sqrt(forecasts["GARCH(1,1)"].clip(lower=EPS))
assert nu>2
for level in [.95,.99]:
    alpha = 1-level
    label = int(round(level*100))
    window = loss.rolling(252,min_periods=200)
    risk[f"VaR_hist_{label}"] = window.quantile(level)
    risk[f"ES_hist_{label}"] = window.apply(lambda v:es_empirico(v,alpha),raw=True)
    q = stats.t.ppf(level,df=nu)
    scale = np.sqrt((nu-2)/nu)
    risk[f"VaR_t_{label}"] = sigma*scale*q
    risk[f"ES_t_{label}"] = sigma*scale*(nu+q*q)/(nu-1)*stats.t.pdf(q,df=nu)/alpha
    cols = [f"{met}_{mod}_{label}" for mod in ["hist","t"] for met in ["VaR","ES"]]
    valid = risk.loc[test,cols].join(loss_d1.rename("loss")).dropna()
    for mod in ["hist","t"]:
        v,es = valid[f"VaR_{mod}_{label}"],valid[f"ES_{mod}_{label}"]
        viola = valid.loss>v
        # Lower pinball loss indicates a better loss quantile forecast.
        error = valid.loss-v
        results_tail.append({"model":mod,"level":level,"common_count":len(valid),
            "violations":int(viola.sum()),"rate_violation":viola.mean(),"rate_esperada":alpha,
            "pinball_VaR":np.maximum(level*error,(level-1)*error).mean(),
            "loss_mean_quando_viola":valid.loss.loc[viola].mean(),
            "ES_mean_quando_viola":es.loc[viola].mean()})
table_risk = pd.DataFrame(results_tail)
display(table_risk)
table_risk.to_csv(OUTPUT/"backtest_VaR_ES.csv",index=False)
risk.loc[test].join(target_date.rename("target_date")).join(loss_d1.rename("loss_observed")).to_csv(
    OUTPUT/"VaR_ES_D1.csv",index_label="origin")
ult = risk.loc[[sessions[-1]]].dropna()
if not ult.empty:
    next_date = cal.next_session(sessions[-1])
    print(f"estimate from the final close in the snapshot: {sessions[-1].date()} → {next_date.date()}")
    last_vol = np.sqrt(forecasts.loc[sessions[-1]].clip(lower=EPS)*252)
    display(last_vol.rename("Forecast annualized volatility").to_frame().style.format("{:.2%}"))
if not ult.empty:
    display(ult.T.rename(columns={ult.index[0]:"daily estimate"}).style.format("{:.2%}"))
fig,ax = plt.subplots(figsize=(13,4))
ax.plot(target_date.loc[test],loss_d1.loc[test],label="Observed loss",lw=.7,color=".5")
for name in ["VaR_hist_95","VaR_t_95","ES_t_95"]:
    ax.plot(target_date.loc[test],risk.loc[test,name],label=name,lw=1)
ax.set_title("Next-session loss tail with 95% VaR and CVaR/ES")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT/"panel_tails.png",dpi=140)
plt.show()
# Central return intervals from a symmetric zero-mean Student t distribution.
intervals = risk.loc[test,["VaR_t_95","VaR_t_99"]].join(r.shift(-1).rename("return_value")).dropna()
bins = []
for tail,central in [(95,.90),(99,.98)]:
    limits = intervals[f"VaR_t_{tail}"]
    bins.append({"coverage_nominal":central,"n":len(limits),
        "coverage_observed":intervals.return_value.abs().le(limits).mean()})
table_bins = pd.DataFrame(bins)
display(table_bins)
table_bins.to_csv(OUTPUT/"coverage_intervals_GARCH.csv",index=False)
fig,ax = plt.subplots(figsize=(13,3.6))
dates = target_date.loc[test]
intervals_plot = intervals.reindex(test)
for tail,alpha_plot in [(99,.16),(95,.25)]:
    lim = intervals_plot[f"VaR_t_{tail}"]
    ax.fill_between(dates,-lim,lim,alpha=alpha_plot,label=f"central interval of {2*tail-100}%")
ax.plot(dates,intervals_plot.return_value,color=".3",lw=.65,label="Observed return")
ax.set_title("Student t GARCH next-session central return intervals")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(ncol=3)
fig.tight_layout()
fig.savefig(OUTPUT/"panel_intervals.png",dpi=140)
plt.show()

## 7. Optional HAR-RV with intraday observations

Set `CSV_RV` to a CSV containing `Date,RV` for the selected asset. RV is the session sum of squared intraday log returns. The notebook checks coverage before fitting.

In [ ]:
if CSV_RV is None:
    print("Optional. Set CSV_RV to evaluate HAR-RV using realized variance from intraday data.")
else:
    pred_rv = walk_forward(rv,pd.DataFrame(index=sessions),test)
    pred_rv = pred_rv.rename(columns={"HAR":"HAR-RV","XGBoost":"XGBoost RV"})
    pred_rv["Mean RV 22"] = rv.rolling(22).mean()
    ranking_rv,common_rv = compare(pred_rv,rv,test)
    display(ranking_rv)
    ranking_rv.to_csv(OUTPUT/"ranking_RV_intraday.csv",index=False)
    panel_rv = common_rv.rename(columns={"observed":"RV_observed_D1"}).join(target_date.rename("target_date"))
    panel_rv.to_csv(OUTPUT/"forecasts_RV_intraday.csv",index_label="origin")

## 8. P1.8 · Matched-date VaR and Expected Shortfall validation

The historical and Student-t GARCH methods are compared only where both have forecasts and an observed loss. Exact binomial tests assess coverage, and paired scores compare the joint tail forecasts.

In [ ]:
# Fallback generated from risk_statistics.py for standalone use in Colab.
try:
    from risk_statistics import session_adjacency, paired_mean_interval, fz0_loss, violation_clustering
except ModuleNotFoundError as error:
    if error.name!="risk_statistics":
        raise
    def session_adjacency(sessions, dates):
        """Mark adjacent exchange sessions while preserving gaps."""
        positions = sessions.get_indexer(dates)
        steps = np.diff(positions)
        if len(positions)==0 or (positions<0).any() or (steps<=0).any():
            raise ValueError("The sample requires known, unique sessions in chronological order.")
        return steps==1

    def paired_mean_interval(difference,adjacent,block=10,rep=3000,seed=42):
        """Apply a circular bootstrap within each contiguous segment, preserving its size."""
        difference = np.asarray(difference,dtype=float)
        adjacent = np.asarray(adjacent,dtype=bool)
        if (difference.ndim!=1 or not len(difference) or not np.isfinite(difference).all()
            or adjacent.shape!=(len(difference)-1,) or block<1 or rep<1):
            raise ValueError("Invalid paired sample, adjacency, or configuration.")
        rng = np.random.default_rng(seed)
        thresholds = np.flatnonzero(~adjacent)+1
        sums = np.zeros(rep)
        for segment in np.split(difference,thresholds):
            k = min(block,len(segment))
            starts = rng.integers(len(segment),size=(rep,int(np.ceil(len(segment)/k))))
            indexes = ((starts[:,:,None]+np.arange(k))%len(segment)).reshape(rep,-1)
            sums += segment[indexes[:,:len(segment)]].sum(axis=1)
        return np.quantile(sums/len(difference),[.025,.975])

    def fz0_loss(loss,var,es,alpha):
        """Joint quantile and tail loss under the positive-loss convention."""
        loss,var,es = (np.asarray(x,dtype=float) for x in (loss,var,es))
        if (not 0<alpha<1 or loss.ndim!=1 or not len(loss) or
            loss.shape!=var.shape or loss.shape!=es.shape or not np.isfinite(np.c_[loss,var,es]).all()):
            raise ValueError("FZ0 requires aligned finite vectors and a tail level between zero and one.")
        if (var<=0).any() or (es<var).any():
            raise ValueError("This FZ0 implementation requires CVaR >= VaR > 0.")
        return np.maximum(loss-var,0)/(alpha*es)+var/es+np.log(es)-1

    def violation_clustering(violations,adjacent,rep=5000,seed=42):
        """One-sided test for excess pairs of adjacent violations."""
        violations = np.asarray(violations,dtype=bool)
        adjacent = np.asarray(adjacent,dtype=bool)
        if violations.ndim!=1 or not len(violations) or adjacent.shape!=(len(violations)-1,) or rep<1:
            raise ValueError("Invalid violations, adjacency, or replication count.")
        esquerda = np.flatnonzero(adjacent)
        total = int(violations.sum())
        pairs = int((violations[esquerda]&violations[esquerda+1]).sum())
        if len(esquerda)==0 or total<2 or total==len(violations):
            return pairs,np.nan
        rng = np.random.default_rng(seed)
        outliers = 0
        for _ in range(rep):
            simulated = np.zeros(len(violations),dtype=bool)
            simulated[rng.choice(len(violations),size=total,replace=False)] = True
            outliers += (simulated[esquerda]&simulated[esquerda+1]).sum()>=pairs
        return pairs,(outliers+1)/(rep+1)

validation_tail,comparison_fz,rows_violations = [],[],[]
for level in [95,99]:
    alpha = (100-level)/100
    cols = [f"{m}_{model}_{level}" for model in ["hist","t"] for m in ["VaR","ES"]]
    sample = risk.loc[test,cols].join(loss_d1.rename("loss")).dropna()
    adjacent = session_adjacency(sessions,sample.index)
    losses = sample.loss.to_numpy()
    scores = {}
    for model in ["hist","t"]:
        v = sample[f"VaR_{model}_{level}"].to_numpy()
        e = sample[f"ES_{model}_{level}"].to_numpy()
        viola = losses>v
        pairs,p_agrupar = violation_clustering(viola,adjacent,seed=SEED)
        total = int(viola.sum())
        coverage = stats.binomtest(total,len(viola),alpha)
        ic = coverage.proportion_ci()
        score = fz0_loss(losses,v,e,alpha)
        scores[model] = score
        validation_tail.append({"level":level,"model":model,"n":len(viola),
            "violations":total,"violations_expected":alpha*len(viola),
            "rate":viola.mean(),"rate_ic_inf":ic.low,"rate_ic_sup":ic.high,
            "p_coverage_binomial":coverage.pvalue,"pairs_disponiveis":int(adjacent.sum()),
            "pairs_violation":pairs,"p_agrupamento":p_agrupar,"FZ0":score.mean()})
        rows_violations.append(pd.DataFrame({"origin":sample.index,"target_date":target_date.loc[sample.index].to_numpy(),
            "level":level,"model":model,"violou":viola,"loss":losses,"VaR":v,"CVaR":e,"FZ0":score}))
    difference = scores["t"]-scores["hist"]
    lo,hi = paired_mean_interval(difference,adjacent,seed=SEED)
    comparison_fz.append({"level":level,"n":len(difference),"FZ0_GARCH_minus_historical":difference.mean(),
                          "IC_inf":lo,"IC_sup":hi})
validation_tail = pd.DataFrame(validation_tail)
comparison_fz = pd.DataFrame(comparison_fz)
display(validation_tail)
display(comparison_fz)
validation_tail.to_csv(OUTPUT/"P1_backtest_coverage_tail.csv",index=False)
comparison_fz.to_csv(OUTPUT/"P1_comparison_VaR_CVaR.csv",index=False)
pd.concat(rows_violations,ignore_index=True).to_csv(OUTPUT/"P1_violations_VaR_por_data.csv",index=False)
import shutil
zip_result = shutil.make_archive(str(OUTPUT),"zip",OUTPUT)
print("Download files:",zip_result)
